# Regression Analysis - Credit Card Data

**Goal:** predict `Amount` (a number) from the features `V1` to `V28`.

**Regression** = predicting a continuous number. (Classification = predicting a category like fraud/legit.)

Low accuracy is expected here, because the V features were not made to predict Amount. This is just practice for the regression techniques.

We use `creditcard_clean.csv` from preprocessing. `Amount` is scaled there, so errors are in scaled units, not real money.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
data = pd.read_csv("creditcard_clean.csv")

v_cols = ["V" + str(i) for i in range(1, 29)]   # V1 ... V28
X = data[v_cols]       # input features
y = data["Amount"]     # target to predict

print(X.shape, y.shape)

## 1. Train-test split

We train the model on 80% of the data and test it on the remaining 20% it has never seen. This tells us how well it works on new data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 2. Correlation with Amount

Find which feature is most related to `Amount`. The best one is used for simple linear regression.

In [ ]:
corr_amt = data[v_cols + ["Amount"]].corr()["Amount"].drop("Amount")

# sort by strength (ignoring + or -)
corr_amt = corr_amt.reindex(corr_amt.abs().sort_values(ascending=False).index)
print(corr_amt.head(8).round(3))

best_feat = corr_amt.index[0]
top5 = corr_amt.index[:5].tolist()
print("Best feature:", best_feat)
print("Top 5 features:", top5)

In [ ]:
plt.figure(figsize=(7, 7))
corr_amt.sort_values().plot(kind="barh")
plt.title("Correlation of V features with Amount")
plt.show()

## Metrics used to judge the models
- **MAE** = average size of the error (lower is better)
- **MSE** = average of squared errors (lower is better)
- **RMSE** = square root of MSE, in the same units as Amount (lower is better)
- **R2** = how much of the variation in Amount the model explains. 1 is perfect, 0 means no better than guessing the average.

In [ ]:
results = []

def evaluate(name, model, Xtr, Xte):
    pred_train = model.predict(Xtr)
    pred_test = model.predict(Xte)
    mse = mean_squared_error(y_test, pred_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred_test),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2 train": r2_score(y_train, pred_train),
        "R2 test": r2_score(y_test, pred_test),
    })

## 3. Simple linear regression (one feature)

Fits a straight line: `Amount = intercept + slope * feature`

In [ ]:
slr = LinearRegression()
slr.fit(X_train[[best_feat]], y_train)

print("Intercept:", round(slr.intercept_, 3))
print("Slope:", round(slr.coef_[0], 3))

evaluate("Simple Linear (" + best_feat + ")", slr, X_train[[best_feat]], X_test[[best_feat]])

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_test[best_feat][:3000], y_test[:3000], s=6, alpha=0.4)

x_line = np.linspace(X_test[best_feat].min(), X_test[best_feat].max(), 100)
y_line = slr.intercept_ + slr.coef_[0] * x_line
plt.plot(x_line, y_line, color="red")

plt.xlabel(best_feat)
plt.ylabel("Amount")
plt.title("Simple linear regression")
plt.show()

## 4. Multiple linear regression (all V features)

Same idea but with many features: `Amount = intercept + b1*V1 + b2*V2 + ... + b28*V28`

In [ ]:
mlr = LinearRegression()
mlr.fit(X_train, y_train)

evaluate("Multiple Linear (V1-V28)", mlr, X_train, X_test)

# 10 biggest coefficients
coef = pd.Series(mlr.coef_, index=v_cols)
coef.reindex(coef.abs().sort_values(ascending=False).index).head(10).round(3)

## 5. Polynomial regression

Adds squared (and cubed) terms so the model can fit curves instead of straight lines. We use only the top 5 features to keep it small, and compare with plain linear regression on the same 5.

In [ ]:
X5_train = X_train[top5]
X5_test = X_test[top5]

# baseline: normal linear regression on top 5
lin5 = LinearRegression()
lin5.fit(X5_train, y_train)
evaluate("Linear (top 5)", lin5, X5_train, X5_test)

# polynomial degree 2 and 3
for degree in [2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    Xp_train = poly.fit_transform(X5_train)
    Xp_test = poly.transform(X5_test)

    model = LinearRegression()
    model.fit(Xp_train, y_train)
    evaluate("Polynomial degree " + str(degree) + " (top 5)", model, Xp_train, Xp_test)

**Overfitting:** if R2 train goes up but R2 test does not, the model is memorising the training data instead of learning a real pattern.

## 6. OLS estimation (statsmodels)

**OLS (Ordinary Least Squares)** is the method that finds the best line by making the sum of squared errors as small as possible. `statsmodels` gives a full statistical report.

In [ ]:
X_sm = sm.add_constant(X_train)   # adds the intercept column
ols = sm.OLS(y_train, X_sm).fit()
print(ols.summary())

**How to read it:**
- `coef` = change in Amount when that feature goes up by 1 (others fixed)
- `P>|t|` (p-value) below 0.05 = that feature is statistically useful
- `R-squared` = share of Amount's variation explained by the model
- `Adj. R-squared` = R-squared adjusted for the number of features

In [ ]:
# only the significant features (p < 0.05)
significant = ols.pvalues[ols.pvalues < 0.05].sort_values()
print("R-squared:", round(ols.rsquared, 4))
print("Adjusted R-squared:", round(ols.rsquared_adj, 4))
print("Significant features:", len(significant) - 1, "(excluding the constant)")
significant.round(4)

## 7. Compare all models

In [ ]:
res = pd.DataFrame(results).set_index("Model").round(4)
res

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
res["RMSE"].plot(kind="barh")
plt.title("RMSE (lower is better)")

plt.subplot(1, 2, 2)
res["R2 test"].plot(kind="barh", color="green")
plt.title("R2 on test data (higher is better)")

plt.tight_layout()
plt.show()

## 8. Residuals

**Residual** = actual value - predicted value. Plotting them shows where the model goes wrong. For a good model, residuals should be small, centred around 0 and spread evenly.

In [ ]:
pred = mlr.predict(X_test)
residuals = y_test - pred

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.scatter(y_test[:3000], pred[:3000], s=6, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs predicted")

plt.subplot(1, 3, 2)
plt.scatter(pred[:3000], residuals[:3000], s=6, alpha=0.4)
plt.axhline(0, color="r", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residuals vs predicted")

plt.subplot(1, 3, 3)
plt.hist(residuals, bins=80)
plt.title("Residual distribution")

plt.tight_layout()
plt.show()

## Conclusion
- R2 values are low, so the V features explain only a small part of Amount.
- Residuals are not evenly spread because Amount is highly skewed. This breaks some OLS assumptions, so read the p-values with caution.
- Polynomial terms improve the fit very little compared to linear regression, so the extra complexity is not worth it.
- Regression is practice here. The real task for this dataset is classification (fraud or not).